# O Gêmeo Digital
#### Disciplina: SME0602 — Motores Numéricos para Simulação em Engenharia
#### Professor: Roberto F. Ausas
#### Grupo 3: 
* #### Beatriz Cosimatti
* #### Cecilia Queiroz
* #### Gabriel Zago
* #### Matheus Buzzon
* #### Pedro Vale
* #### Victor Silva


In [ ]:
# Importações!!!

import sys
import os
import importlib

root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if root not in sys.path:
    sys.path.insert(0, root)

import data_structures
importlib.reload(data_structures)
import analise_falhas
importlib.reload(analise_falhas)
import P2_PARTE_3_GD
importlib.reload(P2_PARTE_3_GD)
import sensitivity_analysis()
importlib.reload(sensitivity_analysis)
import env
importlib.reload(env)
config = env.CONFIG_FALHAS
config_mh = env.CONFIG_MH

##### Nós iniciamos o projeto do gêmeo digital com a rede hidráulica, onde foi possível aplicar os seguintes conceitos de métodos numéricos para estudar as pressões e vazões no sistema de microcanais
- Sistemas lineares em grafos

![img hidraulica](images/h.png)

##### Na placa térmica, utilizamos os seguintes métodos para simular a transmissão de calor:
- Matrizes esparsas
- Iterações de Jacobi, Gauss-Seidel
- Fatoração de Cholesky

![img térmica](images/t.png)

##### Na membrana elástica, usamos os seguintes métodos para entender a vibração da membrana:
- Cálculo de autovalores e autovetores
- Matrizes esparsas
- Diferenciação numérica


![img mecanica](images/m.png)

##### Ao juntar a parte hidráulica com a térmica, foram úteis os seguintes métodos para estudar a influência de uma na outra:
- Métodos de integração
- Análise de Sensibilidade
- Interpolação
- Função de perda 

![img térmico-hidráulica](images/ht.png)

##### Ao juntar a parte hidráulica com a mecânica, foram úteis os seguintes métodos para estudar a influência de uma na outra:
- Aproximação numérica
- Estabilidade/equilibrio
- Relaxação de sistemas

![img mecânico-hidráulica](images/mh.png)


##### Primeiro, usando o método numérico de Monte Carlo, determinamos a probabilidade global de que, após o período de desgaste operacional, a vazão total de entrada 𝑞inlet no circuito (nó 0) seja inferior ao limite crítico de 1.25 × 10−5.

In [ ]:
Xno, conec = data_structures.GeraGrafo(env.CONFIG_FALHAS["LEVELS"]);
Xno = Xno * 0.001

print("=== CALIBRANDO PONTO DE OPERAÇÃO OPERACIONAL ===")
config["INLET_PRESSURE"] = 1.0e4 
vazao_teste = analise_falhas.resolver_vazao_estacionaria(conec, Xno, config, C_estocastico=None)

fator_pressao = 2.0e-5 / vazao_teste
config["INLET_PRESSURE"] = 1.0e4 * fator_pressao

vazao_limpa = analise_falhas.resolver_vazao_estacionaria(conec, Xno, config, C_estocastico=None)
print(f"Pressão calibrada para o ensaio:       {config['INLET_PRESSURE']:.2f} Pa")
print(f"Vazão calculada para a rede sem falhas: {vazao_limpa:.5e} m³/s")
print(f"Limite crítico de falha do enunciado:  {config['V_CRITIC']:.5e} m³/s")
print(f"A rede limpa falha? {vazao_limpa < config['V_CRITIC']}")

##### Considerando agora o acoplamento forte multifísico do modelo completo, estimamos a probabilidade global Prob de que a energia total dissipada atenda ao critério E < 7.0

In [ ]:
analise_falhas.executar_monte_carlo_dinamico(p_O=0.35, f_obs=5, N=2000, plot=True)

##### Depois, utilizando diferentes métodos de aproximação de dados como interpolação e regressão, estudamos o comportamento da potência

In [ ]:
aprox = analise_falhas.aprox_dados(config_mh)
aprox.run()

##### Para avaliar as derivadas parciais do funcional de energia E, da vazão de entrada 𝑞inlet e do volume acumulado de líquido no reservatório ao final do horizonte temporal, 𝑉(𝑡 𝑓 ), em relação a ambos os parâmetros, usamos a seguinte função:

In [ ]:
sensitivity_analysis.prob3()

##### Por fim, ao utilizar o método de Newton-Raphson, determinamos o melhor valor para a largura geométrica H que satisfaça a restrição de projeto para a energia adimensional E = 7.5

In [ ]:
solver_p2 = P2_PARTE_3_GD.P2_3_GD(config_mh)
solver_p2.resolver_P2(E_target=7.5, H0=1000.0e-6, Tc=25.0)